# Gate 4 - Indian-English Strength Sweep

**Purpose:** Run accent conversion at multiple strength levels on Indian-English utterances, measuring identity shift, acoustic quality, and content preservation.

**Metrics per sample per strength:**
- **Mel L1** - spectrogram distance (acoustic quality)
- **Identity shift** - ECAPA-TDNN cosine distance (speaker identity)
- **WER** - faster-whisper transcription (content preservation)

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

---
## Step 1: Environment Setup

In [ ]:
import os, sys, json, time, subprocess, shutil, types, warnings
from pathlib import Path

warnings.simplefilter('ignore')

ACCENTEDGE_DIR = '/content/accentedge'
GATE_DIR = '/content/gate4_artifacts'
DRIVE_BASE = '/content/drive/MyDrive/accentedge/runs'
SAMPLE_RATE = 24000
os.makedirs(GATE_DIR, exist_ok=True)
print(f"Gate dir: {GATE_DIR}")

Install system and pip dependencies.

In [ ]:
!apt-get update -qq && apt-get install -y -qq espeak-ng > /dev/null 2>&1
!pip install -q torchaudio transformers speechbrain faster-whisper \
    phonemizer librosa soundfile pyyaml matplotlib
print("Dependencies installed")

---
## Step 2: Load AccentEdge + FAcodec

In [ ]:
# Clone AccentEdge if not already present
if not os.path.exists(ACCENTEDGE_DIR):
    %cd /content
    !git clone https://github.com/ayushmh/accentedge.git
os.chdir(ACCENTEDGE_DIR)
sys.path.insert(0, f'{ACCENTEDGE_DIR}/src')

print(f"AccentEdge loaded from: {ACCENTEDGE_DIR}")

In [ ]:
# Load FAcodec (Plachta/FAcodec from HuggingFace)
import types as py_types

# Install audiotools mocks (same pattern as gate scripts)
def _mock(name):
    m = py_types.ModuleType(name)
    m.__path__ = []
    m.__package__ = name
    return m

mock_audio = _mock('audiotools')
mock_ml = _mock('audiotools.ml')
mock_ml.BaseModel = type('BaseModel', (), {})
mock_audio.ml = mock_ml
mock_trans = _mock('audiotools.transforms')
mock_audio.transforms = mock_trans
sys.modules['audiotools'] = mock_audio
sys.modules['audiotools.ml'] = mock_ml
sys.modules['audiotools.transforms'] = mock_trans

try:
    from huggingface_hub import hf_hub_download
    from accentedge.codec.facodec import FACodecAdapter

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Download FAcodec checkpoint
    ckpt_path = hf_hub_download(
        repo_id='Plachta/FAcodec',
        filename='vampnet/checkpoint.pth',
        repo_type='model'
    )
    facodec = FACodecAdapter.from_pretrained(
        checkpoint_path=ckpt_path, device=device
    )
    facodec.freeze()
    print(f"FAcodec loaded ({facodec.sample_rate}Hz, {device})")
except Exception as e:
    print(f"FAcodec FAILED: {e}")
    import traceback
    traceback.print_exc()
    facodec = None

In [ ]:
if facodec is None:
    raise RuntimeError(
        "FAcodec is required for Gate 4. "
        "Check audiotools installation and checkpoint availability."
    )

---
## Step 3: Load Phoneme Pipeline

In [ ]:
try:
    from accentedge.phase1.phoneme_pipeline import PhonemePipeline
    FPS = 80
    phone_pipeline = PhonemePipeline(device=device)
    print(f"PhonemePipeline loaded (frame_rate={phone_pipeline.frame_rate_hz}fps)")
except Exception as e:
    print(f"PhonemePipeline FAILED: {e}")
    import traceback
    traceback.print_exc()
    phone_pipeline = None

In [ ]:
if phone_pipeline is None:
    raise RuntimeError('PhonemePipeline is required for Gate 4')

---
## Step 4: Load Identity Evaluator (ECAPA-TDNN)

In [ ]:
try:
    from accentedge.evaluation.identity import IdentityEvaluator
    identity_eval = IdentityEvaluator(device=device)
    print(f"Identity evaluator loaded on {device}")
except Exception as e:
    print(f"Identity evaluator FAILED: {e}")
    import traceback
    traceback.print_exc()
    identity_eval = None

In [ ]:
if identity_eval is None:
    raise RuntimeError('IdentityEvaluator is required for Gate 4')

---
## Step 5: Load Denoiser

The denoiser converts source z_c1 toward target accent. If no checkpoint is provided, the denoiser is randomly initialized (identity shift will be minimal).

In [ ]:
DENOISER_CKPT = os.environ.get('DENOISER_CKPT', '')

try:
    from accentedge.phase1.denoiser import DenoisingTransformerModel
    from accentedge.phase1.converter import AccentConverter

    # Create a denoiser with paper-faithful dimensions
    denoiser = DenoisingTransformerModel(
        d_model=1024, nhead=8, num_layers=6, d_ff=2048,
        phone_vocab_size=393, facodec_dim=8
    ).to(device)

    converter = AccentConverter(
        facodec=facodec,
        denoiser=denoiser,
        phone_pipeline=phone_pipeline,
        device=device
    )

    # Load checkpoint if available
    if DENOISER_CKPT and os.path.exists(DENOISER_CKPT):
        ckpt = torch.load(DENOISER_CKPT, map_location=device, weights_only=False)
        converter.denoiser.load_state_dict(ckpt['denoiser'])
        print(f"Denoiser loaded: {DENOISER_CKPT}")
    else:
        print("WARNING: No denoiser checkpoint - using random weights")
        print(f"  Set DENOISER_CKPT env var or pass checkpoint path")

    converter.freeze_codec()
    print(f"AccentConverter ready on {device}")
except Exception as e:
    print(f"AccentConverter FAILED: {e}")
    import traceback
    traceback.print_exc()
    converter = None

---
## Step 6: Load L2-ARCTIC Indian English Test Data

Download and filter L2-ARCTIC for Hindi L1 speakers (Indian English).

In [ ]:
import torchaudio

try:
    from scripts.dataset_cmu_arctic import L2ArcticDataset
    
    # Download L2-ARCTIC if needed
    L2_ROOT = '/content/l2_arctic'
    if not os.path.exists(L2_ROOT):
        print("Downloading L2-ARCTIC...")
        ds = L2ArcticDataset(root=L2_ROOT)
        ds.download()
    else:
        ds = L2ArcticDataset(root=L2_ROOT)

    # Filter for Hindi speakers
    all_items = ds.all_items()
    hindi_items = [
        item for item in all_items
        if 'Hindi' in (item[2] or '')  # item = (path, speaker, l1)
    ]
    # Note: all_items returns (path, speaker, transcript) for L2Arctic
    # Need to check L1 from speaker metadata
    # For now, filter by speaker ID known to be Hindi
    hindi_speakers = ['HJK', 'HNI', 'HKH', 'HMW', 'HYY']
    hindi_items = [
        item for item in all_items
        if any(spk in item[1] for spk in hindi_speakers)
    ]
    print(f"L2-ARCTIC Hindi speakers: {len(hindi_items)} utterances")
    
    if len(hindi_items) == 0:
        print("WARNING: No Hindi utterances found, using all items")
        hindi_items = all_items
    
except Exception as e:
    print(f"Dataset loading failed: {e}")
    import traceback
    traceback.print_exc()
    hindi_items = []
    ds = None

In [ ]:
import random
random.seed(42)

# Sample N utterances
N_SAMPLES = min(5, len(hindi_items)) if hindi_items else 0
if N_SAMPLES > 0:
    samples = random.sample(hindi_items, N_SAMPLES)
    # Load audio into memory
    loaded_samples = []
    for idx, (path, speaker, transcript) in enumerate(samples):
        wav, sr = torchaudio.load(path)
        if sr != SAMPLE_RATE:
            wav = torchaudio.functional.resample(wav, orig_freq=sr, new_freq=SAMPLE_RATE)
        wav = wav.to(torch.float32)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        loaded_samples.append({
            "id": f"indian_{speaker}_{idx}",
            "speaker": speaker,
            "transcript": transcript,
            "path": path,
            "wav": wav
        })
    print(f"Loaded {len(loaded_samples)} Indian-English utterances")
else:
    loaded_samples = []
    print("No samples loaded")

---
## Step 7: Run Strength Sweep

Run accent conversion at each strength level and measure identity shift, mel L1, and WER.

In [ ]:
import numpy as np

STRENGTHS = [0.0, 0.25, 0.5, 0.75, 1.0]

# Try to import faster-whisper for WER
try:
    from faster_whisper import WhisperModel
    whisper_model = WhisperModel('base', device=device, compute_type='int8')
    HAS_WHISPER = True
    print("faster-whisper loaded for WER")
except Exception as e:
    HAS_WHISPER = False
    print(f"faster-whisper not available: {e}")

def compute_wer(reference: str, hypothesis: str) -> float:
    ref = reference.lower().split()
    hyp = hypothesis.lower().split()
    if len(ref) == 0:
        return 0.0 if len(hyp) == 0 else 1.0
    # Simple WER (word-level edit distance)
    m, n = len(ref), len(hyp)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            if ref[i-1] == hyp[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[m][n] / max(m, 1)

def transcribe(wav_np):
    if not HAS_WHISPER:
        return None
    segments, _ = whisper_model.transcribe(
        wav_np, beam_size=5, language='en', vad_filter=True
    )
    return ' '.join(seg.text for seg in segments).strip()

results = []
t0 = time.time()

for idx, sample in enumerate(loaded_samples):
    sample_id = sample['id']
    speaker = sample['speaker']
    transcript = sample['transcript']
    wav = sample['wav']
    print(f"\n[{idx+1}/{len(loaded_samples)}] {speaker}: {transcript[:50]}...")

    src_np = wav.squeeze().cpu().numpy().astype(np.float32)
    src_sim = identity_eval.similarity(src_np, src_np, sr=SAMPLE_RATE)

    for strength in STRENGTHS:
        try:
            out_wav = converter.convert(wav, transcript, strength=strength)
            out_np = out_wav.squeeze().cpu().numpy().astype(np.float32)

            # Mel L1
            mel_src = librosa.feature.melspectrogram(
                y=src_np, sr=SAMPLE_RATE, n_fft=2048, hop_length=300, n_mels=80
            )
            mel_tgt = librosa.feature.melspectrogram(
                y=out_np, sr=SAMPLE_RATE, n_fft=2048, hop_length=300, n_mels=80
            )
            min_f = min(mel_src.shape[1], mel_tgt.shape[1])
            ml1 = float(np.abs(librosa.power_to_db(mel_src[:, :min_f]) - \
                              librosa.power_to_db(mel_tgt[:, :min_f])).mean())

            # Identity shift
            sim = identity_eval.similarity(src_np, out_np, sr=SAMPLE_RATE)
            identity_shift = 1.0 - sim

            # WER (if available)
            wer = None
            if HAS_WHISPER:
                ref_text = transcribe(src_np) or transcript
                hyp_text = transcribe(out_np)
                if hyp_text is not None:
                    wer = compute_wer(ref_text, hyp_text)

            result = {
                "sample_id": sample_id,
                "speaker": speaker,
                "transcript": transcript,
                "strength": strength,
                "mel_l1": round(ml1, 6),
                "identity_shift": round(identity_shift, 6),
                "wer": round(wer, 6) if wer is not None else None,
                "n_samples": 1,
            }
            results.append(result)
            wer_str = f", wer={wer:.4f}" if wer is not None else ""
            print(f"    s={strength:.2f}: mel_l1={ml1:.4f}, id_shift={identity_shift:.4f}{wer_str}")

        except Exception as e:
            print(f"    s={strength:.2f}: ERROR - {e}")
            results.append({
                "sample_id": sample_id,
                "speaker": speaker,
                "transcript": transcript,
                "strength": strength,
                "mel_l1": None,
                "identity_shift": None,
                "wer": None,
                "error": str(e),
                "n_samples": 0,
            })

elapsed = time.time() - t0
print(f"\nSweep complete in {elapsed:.1f}s")
print(f"Total results: {len(results)}")

---
## Step 8: Save Results

In [ ]:
# Save per-sample results
per_sample_path = f"{GATE_DIR}/metrics_per_sample.json"
with open(per_sample_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Saved: {per_sample_path}")

---
## Step 9: Aggregate Strength Curves

In [ ]:
# Aggregate mean +/- std across samples at each strength
strengths_list = [0.0, 0.25, 0.5, 0.75, 1.0]

curves = {
    "strengths": strengths_list,
    "identity_shift": {"mean": [], "std": [], "n": []},
    "mel_l1": {"mean": [], "std": [], "n": []},
    "wer": {"mean": [], "std": [], "n": []},
}

for s in strengths_list:
    sr = [r for r in results if r['strength'] == s]
    n = len(sr)

    for key in ['identity_shift', 'mel_l1', 'wer']:
        vals = [r[key] for r in sr if r.get(key) is not None]
        if len(vals) > 0:
            curves[key]['mean'].append(round(float(np.mean(vals)), 6))
            curves[key]['std'].append(round(float(np.std(vals)), 6))
            curves[key]['n'].append(len(vals))
        else:
            curves[key]['mean'].append(None)
            curves[key]['std'].append(None)
            curves[key]['n'].append(0)

curves_path = f"{GATE_DIR}/strength_curves.json"
with open(curves_path, 'w') as f:
    json.dump(curves, f, indent=2)
print(f"Saved: {curves_path}")

---
## Step 10: Gate 4 Pass Criteria

In [ ]:
def gate4_check(curves):
    checks = {}

    # C1: identity_shift at s=0 < 0.02
    id0 = curves['identity_shift']['mean'][0]
    checks['id_at_0_lt_002'] = {
        "passed": id0 is not None and id0 < 0.02,
        "value": id0,
        "threshold": 0.02,
        "description": "Identity shift at s=0 < 0.02")

    # C2: identity_shift at s=1 > 0.15
    id1 = curves['identity_shift']['mean'][-1]
    checks['id_at_1_gt_015'] = {
        "passed": id1 is not None and id1 > 0.15,
        "value": id1,
        "threshold": 0.15,
        "description": "Identity shift at s=1 > 0.15")

    # C3: monotonic increase
    id_means = curves['identity_shift']['mean']
    valid = [v for v in id_means if v is not None]
    monotonic = all(valid[i] < valid[i+1] for i in range(len(valid)-1)) if len(valid) > 1 else True
    checks['monotonic'] = {
        "passed": monotonic,
        "value": valid,
        "threshold": "strictly increasing",
        "description": "Identity shift increases with strength")

    # C4: mel_l1 < 0.5 at all strengths
    mel_means = curves['mel_l1']['mean']
    mel_pass = all((v is None or v < 0.5) for v in mel_means)
    checks['mel_l1_lt_05'] = {
        "passed": mel_pass,
        "value": mel_means,
        "threshold": 0.5,
        "description": "Mel L1 < 0.5 at all strengths")

    overall = all(c['passed'] for c in checks.values())
    return {'checks': checks, 'overall': overall}

gate = gate4_check(curves)

# Print results
print("=" * 60)
print("GATE 4 PASS CRITERIA")
print("=" * 60)
for name, c in gate['checks'].items():
    status = 'PASS' if c['passed'] else 'FAIL'
    print(f"[{status}] {c["description"]}")
    print(f"       Value: {c["value"]}, Threshold: {c["threshold"]}")
print("=" * 60)
print(f"GATE 4 OVERALL: {"PASSED" if gate["overall"] else "FAILED"}")

In [ ]:
# Save gate4 manifest
manifest = {
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "device": device,
    "n_samples": len(loaded_samples) if loaded_samples else 0,
    "strengths": strengths_list,
    "curves": curves,
    "gate4_result": gate,
    "facodec_ckpt": "Plachta/FAcodec",
    "denoiser_ckpt": os.environ.get("DENOISER_CKPT", ""),
}

manifest_path = f"{GATE_DIR}/gate4_manifest.json"
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f"Saved: {manifest_path}")

---
## Step 11: Visualize Strength Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Gate 4 - Strength Sweep Curves', fontsize=14, fontweight='bold')

configs = {
    "identity_shift": ("#e74c3c", "Identity Shift (1 - cosine sim)"),
    "mel_l1": ("#3498db", "Mel L1 (dB)"),
    "wer": ("#2ecc71", "Word Error Rate"),
}

for idx, (key, (color, label)) in enumerate(configs.items()):
    ax = axes[idx]
    means = curves[key]['mean']
    stds = curves[key]['std']

    if means[0] is None:
        ax.set_title(f'{label}\n(no data)')
        ax.set_xlabel('Strength')
        continue

    ps = [s for s, m in zip(strengths_list, means) if m is not None]
    pm = [m for m in means if m is not None]
    pstd = [st for st, m in zip(stds, means) if m is not None]

    ax.plot(ps, pm, 'o-', color=color, linewidth=2, markersize=8, label='Mean')
    ax.fill_between(ps,
        [m - st for m, st in zip(pm, pstd)],
        [m + st for m, st in zip(pm, pstd)],
        alpha=0.2, color=color, label='+/- 1 std')

    # Annotate n
    ns = curves[key]['n']
    for s, m, n in zip(strengths_list, means, ns):
        if m is not None and n > 0:
            ax.annotate(f'n={n}', (s, m), textcoords='offset points',
                       xytext=(0, 10), ha='center', fontsize=8)

    ax.set_xlabel('Conversion Strength')
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.set_xticks(strengths_list)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = f"{GATE_DIR}/strength_curves.png"
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"Plot saved: {plot_path}")

from IPython.display import Image, display
display(Image(filename=plot_path))

---
## Step 12: Gate 4 Verdict

Check each criterion and print pass/fail.

In [ ]:
print("=" * 60)
print("GATE 4 VERDICT")
print("=" * 60)
for name, c in gate['checks'].items():
    status = 'PASS' if c['passed'] else 'FAIL'
    print(f"[{status}] {c["description"]}")
print("=" * 60)

overall = gate["overall"]
if overall:
    print("GATE 4 PASSED - All criteria met.")
else:
    print("GATE 4 FAILED - Review criteria above.")

# Save to Drive
drive_out = f'{DRIVE_BASE}/gate4'
os.makedirs(drive_out, exist_ok=True)
for fname in ['metrics_per_sample.json', 'strength_curves.json', 'gate4_manifest.json']:
    src = f'{GATE_DIR}/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, f'{drive_out}/{fname}')
        print(f"  Saved to Drive: {drive_out}/{fname}")
plot_src = f"{GATE_DIR}/strength_curves.png"
if os.path.exists(plot_src):
    shutil.copy2(plot_src, f'{drive_out}/strength_curves.png')
    print(f"  Saved to Drive: {drive_out}/strength_curves.png")

print(f"\nAll artifacts saved to: {drive_out}")

---
## Step 13: Interactive Strength Selector

Select a sample and strength to hear the converted audio.

In [ ]:
from IPython.display import Audio, HTML, display
import tempfile, soundfile as sf

if loaded_samples:
    sample = loaded_samples[0]
    src_np = sample['wav'].squeeze().cpu().numpy().astype(np.float32)
    tmp_src = tempfile.mktemp(suffix='.wav')
    sf.write(tmp_src, src_np, SAMPLE_RATE)

    for target_strength in STRENGTHS:
        out_wav = converter.convert(sample['wav'], sample['transcript'], strength=target_strength)
        out_np = out_wav.squeeze().cpu().numpy().astype(np.float32)
        tmp_conv = tempfile.mktemp(suffix='.wav')
        sf.write(tmp_conv, out_np, SAMPLE_RATE)
        print(f"\nStrength = {target_strength}")
        display(Audio(tmp_conv, autoplay=False, rate=SAMPLE_RATE))
else:
    print("No samples available")

---
### Side-by-Side: Source vs Converted (strength=1.0)

In [ ]:
if loaded_samples:
    sample = loaded_samples[0]
    src_np = sample['wav'].squeeze().cpu().numpy().astype(np.float32)
    tmp_src = tempfile.mktemp(suffix='.wav')
    sf.write(tmp_src, src_np, SAMPLE_RATE)

    out_wav = converter.convert(sample['wav'], sample['transcript'], strength=1.0)
    out_np = out_wav.squeeze().cpu().numpy().astype(np.float32)
    tmp_conv = tempfile.mktemp(suffix='.wav')
    sf.write(tmp_conv, out_np, SAMPLE_RATE)

    dur_src = len(src_np) / SAMPLE_RATE
    dur_conv = len(out_np) / SAMPLE_RATE

    # Build HTML inline
    html = (
        '<div style="display: flex; gap: 20px; flex-wrap: wrap;">'
        '  <div style="flex: 1; min-width: 300px; padding: 15px;'
        '              background: #f8f9fa; border-radius: 8px;'
        '              border: 2px solid #e74c3c;">'
        '    <h3 style="color: #e74c3c; margin-top: 0;">Source (Indian-English)</h3>'
        '    <p><strong>Speaker:</strong> ' + sample['speaker'] + '</p>'
        '    <p><strong>Duration:</strong> {:.2f}s</p>'.format(dur_src)
        '    <p><strong>Transcript:</strong> ' + sample['transcript'][:80] + '</p>'
        '  </div>'
        '  <div style="flex: 1; min-width: 300px; padding: 15px;'
        '              background: #f8f9fa; border-radius: 8px;'
        '              border: 2px solid #3498db;">'
        '    <h3 style="color: #3498db; margin-top: 0;">Converted (strength=1.0)</h3>'
        '    <p><strong>Duration:</strong> {:.2f}s</p>'.format(dur_conv)
        '    <p><strong>Duration ratio:</strong> {:.3f}</p>'.format(dur_conv / max(dur_src, 0.01))
        '  </div>'
        '</div>'
    )

    display(HTML(html))

    print("\nSource audio:")
    display(Audio(tmp_src, autoplay=False, rate=SAMPLE_RATE))

    print("\nConverted audio (strength=1.0):")
    display(Audio(tmp_conv, autoplay=False, rate=SAMPLE_RATE))
else:
    print("No results available")

---
## Step 14: Human Evaluation

Use this section for subjective evaluation of conversion quality.

Listen to each strength level and rate:
1. Naturalness (1-5)
2. Accent strength (1-5)
3. Speaker similarity (1-5)
4. Intelligibility (1-5)

In [ ]:
from IPython.display import display
import pandas as pd

# Create evaluation form template
eval_template = pd.DataFrame(
    columns=['sample_id', 'strength', 'naturalness', 'accent_strength',
             'speaker_similarity', 'intelligibility', 'notes']
)
display(eval_template)

print("\nRecord your evaluations and save to:")
print(f"  {GATE_DIR}/human_evaluation.csv")

---
## Results Summary

The Gate 4 verdict is printed in the cell above. This cell summarizes the key findings:

### Interpretation

A passing Gate 4 result means:
1. No accent change at strength=0 (identity preserved)
2. Significant identity shift at strength=1 (accent applied)
3. Smooth progression between the two extremes
4. Good acoustic quality throughout (low mel L1)

### Next Steps

- Review side-by-side audio for perceptual quality
- If identity shift is too low at s=1, consider stronger denoiser training
- If identity shift is too high at s=0, check FACodec reconstruction quality
- Run Gate 5 (overfit test) to verify denoiser learns valid representations

### Artifacts

All results saved to: `/content/gate4_artifacts/`

- `metrics_per_sample.json` - per-sample-per-strength metrics
- `strength_curves.json` - aggregated curve data for plotting
- `gate4_manifest.json` - run metadata and pass/fail criteria
- `strength_curves.png` - matplotlib visualization